In [12]:
# Cell 0
!pip install -q onnxruntime-gpu==1.26.0
!pip install -q --no-deps insightface 
!pip install -q opencv-python-headless huggingface_hub gfpgan basicsr facexlib fastapi uvicorn python-multipart nest_asyncio

In [13]:
#Cell 1
INSWAPPER_PATH = "/kaggle/input/datasets/ritensinghrawat/faceswap-models/inswapper_128.onnx"
print("Swap model path:", INSWAPPER_PATH)

Swap model path: /kaggle/input/datasets/ritensinghrawat/faceswap-models/inswapper_128.onnx


In [14]:
# Cell 2
GFPGAN_PATH = "/kaggle/input/datasets/ritensinghrawat/faceswap-models/GFPGANv1.4.pth"

import os
assert os.path.exists(INSWAPPER_PATH), "check the exact folder name in the input sidebar"
assert os.path.exists(GFPGAN_PATH), "check the exact folder name in the input sidebar"
print("Models found.")

Models found.


In [15]:
#Cell 3
import os
import importlib.util

spec = importlib.util.find_spec("basicsr")
if spec is not None:
    degradations_path = os.path.join(os.path.dirname(spec.origin), "data", "degradations.py")
    with open(degradations_path, "r") as f:
        content = f.read()
    old_import = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new_import = "from torchvision.transforms.functional import rgb_to_grayscale"
    if old_import in content:
        content = content.replace(old_import, new_import)
        with open(degradations_path, "w") as f:
            f.write(content)
        print("BasicSR patched.")
    else:
        print("BasicSR patch not required.")

BasicSR patch not required.


In [16]:
#Cell 4
import insightface
from insightface.app import FaceAnalysis
from gfpgan import GFPGANer

print("Loading FaceAnalysis...")

providers = [
    "CUDAExecutionProvider",
    "CPUExecutionProvider"
]

face_app = FaceAnalysis(
    name="buffalo_l",
    providers=providers
)

face_app.prepare(
    ctx_id=0,
    det_size=(640, 640)
)

print("FaceAnalysis loaded.")


print("Loading face swapper...")

swapper = insightface.model_zoo.get_model(
    INSWAPPER_PATH,
    download=False,
    providers=providers
)

print("Face swapper loaded.")


print("Loading GFPGAN...")

restorer = GFPGANer(
    model_path=GFPGAN_PATH,
    upscale=1,
    arch="clean",
    channel_multiplier=2
)

print("GFPGAN loaded.")

print()
print("====================================")
print("ALL MODELS LOADED")
print("====================================")

Loading FaceAnalysis...
FaceAnalysis loaded.
Loading face swapper...
inswapper-shape: [1, 3, 128, 128]
Face swapper loaded.
Loading GFPGAN...
GFPGAN loaded.

ALL MODELS LOADED


In [17]:
#Cell 5
def restore_face(image):

    _, _, restored_img = restorer.enhance(
        image,
        has_aligned=False,
        only_center_face=False,
        paste_back=True
    )

    return restored_img

In [18]:
#Cell 6
def swap_and_restore(
    source_path,
    target_path,
    output_path="result.jpg"
):

    # -----------------------------------------
    # Read images
    # -----------------------------------------

    source_img = cv2.imread(source_path)
    target_img = cv2.imread(target_path)

    if source_img is None:
        raise ValueError(
            "Could not read source image"
        )

    if target_img is None:
        raise ValueError(
            "Could not read target image"
        )


    # -----------------------------------------
    # Detect faces
    # -----------------------------------------

    source_faces = face_app.get(source_img)
    target_faces = face_app.get(target_img)

    if not source_faces:
        raise ValueError(
            "No face detected in source image"
        )

    if not target_faces:
        raise ValueError(
            "No face detected in target image"
        )


    # -----------------------------------------
    # Use first source face
    # -----------------------------------------

    source_face = source_faces[0]


    # -----------------------------------------
    # Perform face swap
    # -----------------------------------------

    result = target_img.copy()

    for face in target_faces:

        result = swapper.get(
            result,
            face,
            source_face,
            paste_back=True
        )


    # -----------------------------------------
    # Restore face
    # -----------------------------------------

    result = restore_face(result)


    # -----------------------------------------
    # Save result
    # -----------------------------------------

    success = cv2.imwrite(
        output_path,
        result
    )

    if not success:
        raise ValueError(
            "Could not save output image"
        )

    return output_path

In [19]:
#Cell 7
# import cv2

# SOURCE_IMAGE = (
#     "/kaggle/input/datasets/ritensinghrawat/your-test-images/images.jpg"
# )

# TARGET_IMAGE = (
#     "/kaggle/input/datasets/ritensinghrawat/your-test-images/Rohan.jpeg"
# )

# TEST_OUTPUT = "/kaggle/working/test_result.jpg"

# result_path = swap_and_restore(
#     source_path=SOURCE_IMAGE,
#     target_path=TARGET_IMAGE,
#     output_path=TEST_OUTPUT
# )

# print("Result saved to:")
# print(result_path)

In [20]:
# #Cell 8
# from IPython.display import Image, display

# display(
#     Image(
#         filename="/kaggle/working/test_result.jpg"
#     )
# )

In [21]:
#Cell 9
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse, JSONResponse

import shutil
import uuid
import os

app = FastAPI(
    title="Face Swap API",
    description="InsightFace + GFPGAN Face Swap API",
    version="1.0.0"
)

print("FastAPI application created.")

FastAPI application created.


In [22]:
#Cell 10
@app.get("/")
async def root():

    return {
        "status": "online",
        "service": "Face Swap API",
        "version": "1.0.0"
    }


@app.get("/health")
async def health():

    return {
        "status": "healthy"
    }


@app.post("/swap-face")
async def swap_face_endpoint(
    source: UploadFile = File(...),
    target: UploadFile = File(...)
):

    request_id = str(uuid.uuid4())

    source_path = (
        f"/kaggle/working/"
        f"{request_id}_source.jpg"
    )

    target_path = (
        f"/kaggle/working/"
        f"{request_id}_target.jpg"
    )

    output_path = (
        f"/kaggle/working/"
        f"{request_id}_output.jpg"
    )


    try:

        # -----------------------------------------
        # Save source image
        # -----------------------------------------

        with open(source_path, "wb") as f:

            shutil.copyfileobj(
                source.file,
                f
            )


        # -----------------------------------------
        # Save target image
        # -----------------------------------------

        with open(target_path, "wb") as f:

            shutil.copyfileobj(
                target.file,
                f
            )


        # -----------------------------------------
        # Run face swap
        # -----------------------------------------

        swap_and_restore(
            source_path=source_path,
            target_path=target_path,
            output_path=output_path
        )


        # -----------------------------------------
        # Return generated image
        # -----------------------------------------

        return FileResponse(
            output_path,
            media_type="image/jpeg",
            filename="result.jpg"
        )


    except Exception as e:

        return JSONResponse(
            status_code=500,
            content={
                "error": str(e)
            }
        )


    finally:

        # -----------------------------------------
        # Remove temporary input images
        # -----------------------------------------

        if os.path.exists(source_path):
            os.remove(source_path)

        if os.path.exists(target_path):
            os.remove(target_path)

In [23]:
#Cell 11
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("====================================")
print("🚀 FACE SWAP API STARTING")
print("====================================")
print()
print("Local API:")
print("http://127.0.0.1:8000")
print()
print("Swagger:")
print("http://127.0.0.1:8000/docs")
print()

🚀 FACE SWAP API STARTING

Local API:
http://127.0.0.1:8000

Swagger:
http://127.0.0.1:8000/docs



In [24]:
# #Cell 12
# import requests
# import time

# time.sleep(3)

# response = requests.get(
#     "http://127.0.0.1:8000/health"
# )

# print("Status code:", response.status_code)
# print("Response:", response.json())

In [25]:
#Cell 13
!wget -q \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
-O /kaggle/working/cloudflared.deb

!dpkg -i /kaggle/working/cloudflared.deb

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Selecting previously unselected package cloudflared.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../kaggle/working/cloudflared.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [26]:
#Cell 14
!cloudflared --version

cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


In [27]:
#Cell 15
import subprocess
import re
import time

print("Starting Cloudflare Tunnel...")
print()

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None

start_time = time.time()

while time.time() - start_time < 60:

    line = tunnel_process.stdout.readline()

    if not line:
        continue

    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:

        public_url = match.group(0)

        break


print()
print("============================================")
print("🚀 PUBLIC FACE SWAP API")
print("============================================")

if public_url:

    print()
    print("API:")
    print(public_url)

    print()
    print("Health:")
    print(public_url + "/health")

    print()
    print("Swagger:")
    print(public_url + "/docs")

else:

    print("⚠️ Could not find the Cloudflare URL.")
    print("Check the tunnel output above.")

Starting Cloudflare Tunnel...

2026-09-16T13:14:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T13:14:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T13:14:43Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T13:14:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T13:14:43Z INF |  https://listprice-newf

In [32]:
#Cell 17
import cv2
import requests
from IPython.display import Image, display
if public_url:

    with open(
        SOURCE_IMAGE,
        "rb"
    ) as source_file, open(
        TARGET_IMAGE,
        "rb"
    ) as target_file:

        response = requests.post(

            public_url + "/swap-face",

            files={
                "source": (
                    "source.jpg",
                    source_file,
                    "image/jpeg"
                ),

                "target": (
                    "target.jpg",
                    target_file,
                    "image/jpeg"
                )
            }
        )


    print("Status:", response.status_code)


    if response.status_code == 200:

        PUBLIC_TEST_OUTPUT = (
            "/kaggle/working/"
            "public_api_result.jpg"
        )

        with open(
            PUBLIC_TEST_OUTPUT,
            "wb"
        ) as f:

            f.write(response.content)


        print(
            "Result saved:",
            PUBLIC_TEST_OUTPUT
        )


    else:

        print("API Error:")
        print(response.text)

else:

    print("No public URL available.")

INFO:     34.86.7.55:0 - "POST /swap-face HTTP/1.1" 200 OK
Status: 200
Result saved: /kaggle/working/public_api_result.jpg
